<a href="https://colab.research.google.com/github/solucionesmatematicas27-code/Actividad/blob/main/generar_video_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Genera simulacion_entrenamiento_red_neuronal.mp4
# Entrena en NumPy una red 2-4-4-1 con Adam (mismos datos y semillas que el .tex) y renderiza el video.
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.animation import FFMpegWriter
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import make_moons

# ---------- datos y red (igual que en el .tex) ----------
rng = np.random.default_rng(7)
X, y = make_moons(n_samples=120, noise=0.15, random_state=3)
X = (X - X.mean(0)) / X.std(0)
y = y.reshape(-1, 1).astype(float)
sizes = [2, 4, 4, 1]
W = [rng.normal(0, 0.8, (sizes[i], sizes[i + 1])) for i in range(3)]
b = [np.zeros((1, sizes[i + 1])) for i in range(3)]

def forward(Xin, W, b):
    a1 = np.tanh(Xin @ W[0] + b[0])
    a2 = np.tanh(a1 @ W[1] + b[1])
    p = 1 / (1 + np.exp(-(a2 @ W[2] + b[2])))
    return a1, a2, p

def perdida(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

P = W + b
m = [np.zeros_like(q) for q in P]
v = [np.zeros_like(q) for q in P]
lr, B1, B2, EPS = 0.03, 0.9, 0.999, 1e-8
EPOCAS = 300
g = np.linspace(-2.5, 2.5, 90)
GX, GY = np.meshgrid(g, g)
malla = np.c_[GX.ravel(), GY.ravel()]

estados = []   # (pesos, prob_malla, perdida, exactitud)
for t in range(EPOCAS + 1):
    a1, a2, p = forward(X, P[:3], P[3:])
    estados.append(([w.copy() for w in P[:3]],
                    forward(malla, P[:3], P[3:])[2].reshape(GX.shape),
                    perdida(p), float(np.mean((p > .5) == (y > .5)))))
    if t == EPOCAS:
        break
    n = len(X)
    d3 = (p - y) / n
    gW2 = a2.T @ d3; gb2 = d3.sum(0, keepdims=True)
    d2 = (d3 @ P[2].T) * (1 - a2 ** 2)
    gW1 = a1.T @ d2; gb1 = d2.sum(0, keepdims=True)
    d1 = (d2 @ P[1].T) * (1 - a1 ** 2)
    gW0 = X.T @ d1; gb0 = d1.sum(0, keepdims=True)
    G = [gW0, gW1, gW2, gb0, gb1, gb2]
    k = t + 1
    for i in range(6):
        m[i] = B1 * m[i] + (1 - B1) * G[i]
        v[i] = B2 * v[i] + (1 - B2) * G[i] ** 2
        P[i] = P[i] - lr * (m[i] / (1 - B1 ** k)) / (np.sqrt(v[i] / (1 - B2 ** k)) + EPS)

# ---------- figura ----------
VERDE, ROJO, AZUL = "#22693C", "#961E1E", "#1B2A6B"
fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.16], width_ratios=[1.25, 1, 1],
                      left=0.04, right=0.97, top=0.86, bottom=0.05, wspace=0.18, hspace=0.12)
axN = fig.add_subplot(gs[0, 0]); axD = fig.add_subplot(gs[0, 1]); axL = fig.add_subplot(gs[0, 2])
axT = fig.add_subplot(gs[1, :]); axT.axis("off")
fig.suptitle("Simulación del entrenamiento de una red neuronal con Adam",
             fontsize=19, fontweight="bold", color=AZUL, y=0.96)

# red
xs = [0, 1, 2, 3]
ys = {0: [0.6, -0.6], 1: [1.5, 0.5, -0.5, -1.5], 2: [1.5, 0.5, -0.5, -1.5], 3: [0.0]}
segs, idx = [], []
for L in range(3):
    for i, yi in enumerate(ys[L]):
        for j, yj in enumerate(ys[L + 1]):
            segs.append([(xs[L], yi), (xs[L + 1], yj)]); idx.append((L, i, j))
lc = LineCollection(segs, linewidths=1, colors="gray")
axN.add_collection(lc)
for L in range(4):
    axN.scatter([xs[L]] * len(ys[L]), ys[L], s=420, c="white", edgecolors="#333", linewidths=1.5, zorder=3)
for L, nom in enumerate(["Entrada", "Oculta 1", "Oculta 2", "Salida"]):
    axN.text(xs[L], 2.05, nom, ha="center", fontsize=11)
axN.set_xlim(-0.35, 3.35); axN.set_ylim(-2.0, 2.4); axN.axis("off")
axN.set_title("Pesos de la red\n(azul > 0, rojo < 0, grosor = magnitud)", fontsize=11)

# frontera
cmap = LinearSegmentedColormap.from_list("pb", ["#8aa4ff", "#ffffff", "#ff8a8a"])
im = axD.imshow(estados[0][1], extent=[-2.5, 2.5, -2.5, 2.5], origin="lower",
                cmap=cmap, vmin=0, vmax=1, interpolation="bilinear", zorder=0)
axD.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], c="#1F3FBF", s=28, edgecolors="white", linewidths=.5, zorder=2)
axD.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], c="#C01F1F", s=28, edgecolors="white", linewidths=.5, zorder=2)
axD.set_xlim(-2.5, 2.5); axD.set_ylim(-2.5, 2.5); axD.set_xticks([]); axD.set_yticks([])
axD.set_title("Frontera de decisión", fontsize=11)

# pérdida
perd = [s[2] for s in estados]
axL.set_xlim(0, EPOCAS); axL.set_ylim(0, max(perd) * 1.05)
axL.set_xlabel("época"); axL.set_ylabel("pérdida (entropía cruzada)")
axL.set_title("Pérdida de entrenamiento", fontsize=11); axL.grid(alpha=.3)
(linea,) = axL.plot([], [], color=VERDE, lw=2.2)
(punto,) = axL.plot([], [], "o", color=ROJO, ms=7)

txt = axT.text(0.0, 0.62, "", fontsize=16, fontweight="bold", color="#222", va="center")
sub = axT.text(0.0, 0.0, "", fontsize=11.5, color="#555", va="center")

def pasos(t):
    return ("1) forward: los datos atraviesan la red   2) pérdida: se mide el error   "
            "3) retropropagación: se calculan los gradientes   4) Adam: se actualizan los pesos")

def dibujar(t):
    Wc, prob, ls, ac = estados[t]
    cols, lws = [], []
    for (L, i, j) in idx:
        w = Wc[L][i, j]
        a = min(abs(w) / 2.5, 1.0)
        base = np.array([0.12, 0.37, 0.75]) if w > 0 else np.array([0.75, 0.22, 0.17])
        cols.append((*base, 0.25 + 0.7 * a)); lws.append(0.4 + 5.5 * a)
    lc.set_color(cols); lc.set_linewidth(lws)
    im.set_data(prob)
    linea.set_data(range(t + 1), perd[:t + 1])
    punto.set_data([t], [perd[t]])
    txt.set_text(f"Época {t:3d} / {EPOCAS}     pérdida = {ls:.3f}     exactitud = {100 * ac:.1f} %")
    sub.set_text(pasos(t))

# horario: lento al inicio, rápido después
orden = []
for t in range(EPOCAS + 1):
    rep = 4 if t <= 30 else (2 if t <= 100 else 1)
    orden += [t] * rep
orden += [EPOCAS] * 45   # pausa final

writer = FFMpegWriter(fps=30, codec="libx264", bitrate=3000,
                      extra_args=["-pix_fmt", "yuv420p"],
                      metadata={"title": "Entrenamiento de una red neuronal con Adam"})
with writer.saving(fig, "simulacion_entrenamiento_red_neuronal.mp4", dpi=100):
    previo = None
    for t in orden:
        if t != previo:
            dibujar(t); previo = t
        writer.grab_frame()
print("cuadros:", len(orden), "duracion (s):", round(len(orden) / 30, 1))
print("perdida final:", estados[-1][2], "exactitud final:", estados[-1][3])

cuadros: 509 duracion (s): 17.0
perdida final: 0.02795805731650784 exactitud final: 0.9916666666666667
